# Convolutional Hybrid QNN — Alternative Fusion Methods Benchmark

Benchmarks three fusion strategies as alternatives to FFT-based frequency-domain fusion:

| Step | Method | Description |
|------|--------|-------------|
| 2 | **RFF** (Random Fourier Features) | Kernel-space inner product via Bochner's theorem. Near-FFT baseline. |
| 3 | **Cross-Attention** | Learns which classical/quantum interactions matter. Dominant modern fusion. |
| 4 | **Bilinear / Tensor Product** | Outer product c⊗q captures all pairwise cross-feature interactions. |

**Sweep axes:** `fusion_type` × `quantum_type` (+ `kernel` for RFF) = **8 configs**

**References (from other notebooks):**
- FiLM baseline (PauliZ expvals): **0.9566** macro F1
- FFT best (complex + product + gelu): **0.9717** macro F1

In [1]:
import os
import random
import copy
import json
import time
import math
import itertools
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from scipy.signal import get_window
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | Seed: {SEED}')

Device: cuda | Seed: 42


In [2]:
# ==========================================
# DATASET CONFIGURATION  (unchanged)
# ==========================================
BASE_DIR = Path('/home/sammarv/quantum_corrosion')

TARGET_CLASSES = [
    {'id': 0, 'name': 'Spread 0.5g', 'path_sub': 'data/raw/0.5/0.5.iq',  'gram': 0.5},
    {'id': 1, 'name': 'Spread 1.0g', 'path_sub': 'data/raw/1/1.iq',      'gram': 1.0},
    {'id': 2, 'name': 'Spread 1.5g', 'path_sub': 'data/raw/1.5/1.5.iq',  'gram': 1.5},
    {'id': 3, 'name': 'Spread 2.0g', 'path_sub': 'data/raw/2/2.iq',      'gram': 2.0},
    {'id': 4, 'name': 'Spread 2.5g', 'path_sub': 'data/raw/2.5/2.5.iq',  'gram': 2.5},
]

N_CLASSES   = len(TARGET_CLASSES)
LABEL_NAMES = [tc['name'] for tc in TARGET_CLASSES]

OUT_DIR = BASE_DIR / 'results/spread_conv_hybrid_qnn_fusion_alternatives'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FFT_SIZE        = 4096
N_STACKS        = 16
OVERLAP         = 0.25
HOP             = int(FFT_SIZE * (1 - OVERLAP))
SAMPLES_PER_IMG = N_STACKS * HOP + (FFT_SIZE - HOP)
WIN             = get_window('hann', FFT_SIZE).astype(np.float32)

iqs = {}
for tc in TARGET_CLASSES:
    p = BASE_DIR / tc['path_sub']
    iq = np.memmap(str(p), dtype='complex64', mode='r')
    n_imgs = len(iq) // SAMPLES_PER_IMG
    iqs[tc['id']] = iq
    print(f"  label={tc['id']}  {tc['name']:14s}  images={n_imgs:,}")

  label=0  Spread 0.5g     images=9,585
  label=1  Spread 1.0g     images=8,666
  label=2  Spread 1.5g     images=9,141
  label=3  Spread 2.0g     images=10,416
  label=4  Spread 2.5g     images=10,643


In [3]:
# ==========================================
# FEATURE EXTRACTION  (unchanged)
# ==========================================

def extract_sota_features(iq_mmap, img_idx):
    start = img_idx * SAMPLES_PER_IMG
    end   = start + SAMPLES_PER_IMG
    if end > len(iq_mmap):
        return None
    frames = np.empty((N_STACKS, FFT_SIZE), dtype='complex64')
    for i in range(N_STACKS):
        s = start + i * HOP
        frames[i] = iq_mmap[s : s + FFT_SIZE]
    spec     = np.fft.fftshift(np.fft.fft(frames * WIN, axis=1), axes=1)
    mag      = np.abs(spec).astype(np.float32)
    spec_db  = 20.0 * np.log10(mag + 1e-12)
    spec_512 = spec_db.reshape(N_STACKS, 512, 8).mean(axis=2)
    return spec_512


def extract_class(label, n_samples, n_workers=16, desc=''):
    iq     = iqs[label]
    n_imgs = len(iq) // SAMPLES_PER_IMG
    idxs   = np.random.choice(n_imgs, size=n_samples, replace=(n_samples > n_imgs))
    features, labels_out = [], []

    def _worker(idx):
        spec = extract_sota_features(iq, int(idx))
        if spec is None:
            return None, None
        sig_std = spec.std()
        noise   = np.random.randn(*spec.shape).astype(np.float32) * (sig_std * 0.05)
        return spec, spec + noise

    t0 = time.time()
    done = 0
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_worker, idx): idx for idx in idxs}
        for fut in as_completed(futs):
            orig, aug = fut.result()
            if orig is not None:
                features.append(orig)
                features.append(aug)
                labels_out.extend([label, label])
            done += 1
            if done % 500 == 0 or done == len(idxs):
                print(f'  [{desc}] {done}/{len(idxs)}  {time.time()-t0:.0f}s')
    return features, labels_out


print('Feature extraction utilities ready.')

Feature extraction utilities ready.


In [4]:
# ==========================================
# BUILD DATASET  (shared cache with FFT notebook)
# ==========================================
cache_dir = BASE_DIR / 'results/spread_conv_hybrid_qnn/feature_cache'
cache_dir.mkdir(parents=True, exist_ok=True)

DATASET_FRACTION       = 0.10
SAMPLES_PER_CLASS_BASE = 3000
SAMPLES_PER_CLASS      = max(1, int(SAMPLES_PER_CLASS_BASE * DATASET_FRACTION))
fraction_tag           = f"{int(DATASET_FRACTION * 100)}pct"

X_cache = cache_dir / f'X_spec_{fraction_tag}.npy'
y_cache = cache_dir / f'y_{fraction_tag}.npy'

if X_cache.exists() and y_cache.exists():
    print(f'Loading {fraction_tag} features from cache...')
    X_all = np.load(X_cache)
    y_all = np.load(y_cache)
else:
    print('Extracting features...')
    all_feats, all_labels = [], []
    for tc in TARGET_CLASSES:
        feats, lbls = extract_class(tc['id'], SAMPLES_PER_CLASS, desc=tc['name'])
        all_feats.extend(feats)
        all_labels.extend(lbls)
    X_all = np.stack(all_feats).astype(np.float32)
    y_all = np.array(all_labels, dtype=np.int64)
    np.nan_to_num(X_all, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    np.save(X_cache, X_all)
    np.save(y_cache, y_all)

print(f'Dataset: {X_all.shape}  labels: {np.bincount(y_all)}')

Loading 10pct features from cache...
Dataset: (3000, 16, 512)  labels: [600 600 600 600 600]


In [5]:
# ==========================================
# PREPROCESSING & DATA LOADERS  (unchanged)
# ==========================================
N, H, W  = X_all.shape
X_flat   = X_all.reshape(N, -1)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_flat).reshape(N, H, W).astype(np.float32)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_scaled, y_all, test_size=0.20, random_state=SEED, stratify=y_all
)

X_train_t = torch.tensor(X_train_np[:, None, :, :], dtype=torch.float32)
y_train_t = torch.tensor(y_train_np, dtype=torch.long)
X_test_t  = torch.tensor(X_test_np[:, None, :, :],  dtype=torch.float32)
y_test_t  = torch.tensor(y_test_np,  dtype=torch.long)

batch_size = 32
_n_workers = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=batch_size, shuffle=True, drop_last=True,
    num_workers=_n_workers, pin_memory=True, persistent_workers=True,
)
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=batch_size, shuffle=False,
    num_workers=_n_workers, pin_memory=True, persistent_workers=True,
)

print(f'Train: {X_train_np.shape}  Test: {X_test_np.shape}')
print(f'Train batches: {len(train_loader)}  Test batches: {len(test_loader)}')

Train: (2400, 16, 512)  Test: (600, 16, 512)
Train batches: 75  Test batches: 19


In [6]:
# ==========================================
# QUANTUM DEVICES & CIRCUITS  (unchanged)
# ==========================================
n_qubits = 4
n_layers = 4

def _make_device(n_q, prefer_gpu=True):
    if prefer_gpu:
        try:
            d = qml.device('lightning.gpu', wires=n_q)
            print(f'  {d.name}')
            return d, 'adjoint'
        except Exception:
            pass
    d = qml.device('default.qubit', wires=n_q)
    print(f'  {d.name}')
    return d, 'backprop'

print('Quantum device (real qnode):')
dev_real, diff_real = _make_device(n_qubits)

print('Quantum device (complex qnode):')
dev_complex = qml.device('default.qubit', wires=n_qubits)


@qml.qnode(dev_real, interface='torch', diff_method=diff_real)
def qnode_real(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


@qml.qnode(dev_complex, interface='torch', diff_method='backprop')
def qnode_complex(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.state()


print(f'\nn_qubits={n_qubits}  n_layers={n_layers}')
print(f'Real output dim : {n_qubits}')
print(f'Complex output dim: 2 * 2^{n_qubits} = {2 * 2**n_qubits}  (real+imag split)')

Quantum device (real qnode):
  Default qubit PennyLane plugin
Quantum device (complex qnode):

n_qubits=4  n_layers=4
Real output dim : 4
Complex output dim: 2 * 2^4 = 32  (real+imag split)


In [7]:
# ==========================================
# BATCHED QUANTUM LAYERS  (unchanged)
# ==========================================

class BatchedQuantumLayer(nn.Module):
    def __init__(self, n_l, n_q, q_node):
        super().__init__()
        self.n_qubits = n_q
        self.qnode    = q_node
        self.weights  = nn.Parameter(torch.empty(n_l, n_q).uniform_(-np.pi, np.pi))

    def forward(self, x):
        B   = x.shape[0]
        out = self.qnode(x, self.weights)
        return out.float().view(self.n_qubits, B).t().contiguous()


class BatchedQuantumLayerComplex(nn.Module):
    def __init__(self, n_l, n_q, q_node):
        super().__init__()
        self.n_qubits  = n_q
        self.state_dim = 2 ** n_q
        self.qnode     = q_node
        self.weights   = nn.Parameter(torch.empty(n_l, n_q).uniform_(-np.pi, np.pi))

    def forward(self, x):
        B   = x.shape[0]
        out = self.qnode(x, self.weights)
        if out.dim() == 1:
            out = out.unsqueeze(0)
        if out.shape[0] != B:
            out = out.t()
        q_real = out.real.float()
        q_imag = out.imag.float()
        return torch.cat([q_real, q_imag], dim=-1).contiguous()


print('Batched quantum layers ready.')

Batched quantum layers ready.


In [8]:
# ==========================================
# ALTERNATIVE FUSION MODULES
# ==========================================

class RFFusion(nn.Module):
    """
    Random Fourier Feature fusion (Bochner's theorem).

    Pipeline:
      c (B, c_dim) ──proj_c──► RFF map ─┐
                                          ├─ element-wise product ─► LayerNorm ─► GELU ─► classifier
      q (B, q_dim) ──proj_q──► RFF map ─┘

    z(x) = sqrt(2/D) * cos(x @ W.T + b) approximates a shift-invariant kernel:
      kernel='rbf'       : W ~ N(0, 2*gamma)    — Gaussian / RBF kernel
      kernel='laplacian' : W ~ Cauchy(0, gamma)  — Laplacian kernel

    z(c) ⊙ z(q) approximates k(c, q): similarity in kernel space.
    W and b are fixed random buffers (not learned).
    """
    def __init__(self, c_dim, q_dim, d_rff=64, n_classes=5, kernel='rbf', gamma=1.0):
        super().__init__()
        self.d_rff  = d_rff
        self.kernel = kernel

        self.proj_c = nn.Linear(c_dim, d_rff)
        self.proj_q = nn.Linear(q_dim, d_rff)

        if kernel == 'rbf':
            W = torch.randn(d_rff, d_rff) * math.sqrt(2.0 * gamma)
        else:  # laplacian
            W = torch.distributions.Cauchy(
                torch.zeros(d_rff, d_rff),
                torch.full((d_rff, d_rff), gamma)
            ).sample()
        b = torch.rand(d_rff) * 2.0 * math.pi

        self.register_buffer('W', W)
        self.register_buffer('b', b)

        self.norm       = nn.LayerNorm(d_rff)
        self.act        = nn.GELU()
        self.classifier = nn.Linear(d_rff, n_classes)

    def _rff(self, x):
        return math.sqrt(2.0 / self.d_rff) * torch.cos(x @ self.W.t() + self.b)

    def forward(self, c, q):
        c_proj = self.proj_c(c)       # (B, d_rff)
        q_proj = self.proj_q(q)       # (B, d_rff)
        z_c    = self._rff(c_proj)    # (B, d_rff)
        z_q    = self._rff(q_proj)    # (B, d_rff)
        F      = z_c * z_q            # (B, d_rff) — approx kernel inner product
        return self.classifier(self.act(self.norm(F)))


class CrossAttnFusion(nn.Module):
    """
    Bidirectional cross-attention fusion.

    c attends to q (classical queries quantum) and q attends to c (quantum queries classical).
    Outputs are concatenated, projected to d_model, normalised, then classified.
    Strictly more expressive than element-wise fusion: captures asymmetric interactions
    and learns which feature components matter for each query.
    """
    def __init__(self, c_dim, q_dim, d_model=64, n_heads=4, n_classes=5, dropout=0.1):
        super().__init__()
        self.proj_c    = nn.Linear(c_dim, d_model)
        self.proj_q    = nn.Linear(q_dim, d_model)
        self.attn_cq   = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_qc   = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.proj_fuse = nn.Linear(2 * d_model, d_model)
        self.norm      = nn.LayerNorm(d_model)
        self.act       = nn.GELU()
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, c, q):
        c_tok = self.proj_c(c).unsqueeze(1)              # (B, 1, d_model)
        q_tok = self.proj_q(q).unsqueeze(1)              # (B, 1, d_model)
        attn_c, _ = self.attn_cq(c_tok, q_tok, q_tok)   # c attends to q
        attn_q, _ = self.attn_qc(q_tok, c_tok, c_tok)   # q attends to c
        fused = torch.cat([attn_c.squeeze(1), attn_q.squeeze(1)], dim=-1)  # (B, 2*d_model)
        fused = self.act(self.norm(self.proj_fuse(fused)))
        return self.classifier(fused)


class BilinearFusion(nn.Module):
    """
    Tensor product / bilinear fusion.

    Outer product c_proj ⊗ q_proj → (B, d_bp, d_bp) captures all pairwise
    cross-feature interactions (classical feature i with quantum feature j for all i,j).
    Flattened to d_bp² and classified. With d_bp=32: 1024-dim fused representation.
    """
    def __init__(self, c_dim, q_dim, d_bp=32, n_classes=5):
        super().__init__()
        self.proj_c    = nn.Linear(c_dim, d_bp)
        self.proj_q    = nn.Linear(q_dim, d_bp)
        self.norm      = nn.LayerNorm(d_bp * d_bp)
        self.act       = nn.GELU()
        self.classifier = nn.Linear(d_bp * d_bp, n_classes)

    def forward(self, c, q):
        c_proj = self.proj_c(c)                               # (B, d_bp)
        q_proj = self.proj_q(q)                               # (B, d_bp)
        outer  = torch.einsum('bi,bj->bij', c_proj, q_proj)   # (B, d_bp, d_bp)
        flat   = outer.flatten(1)                              # (B, d_bp²)
        return self.classifier(self.act(self.norm(flat)))


print('RFFusion, CrossAttnFusion, BilinearFusion defined.')

RFFusion, CrossAttnFusion, BilinearFusion defined.


In [9]:
# ==========================================
# UNIFIED HYBRID QNN WITH PLUGGABLE FUSION
# ==========================================

class HybridQNN_Alt(nn.Module):
    """
    Same CNN backbone and quantum circuits as HybridQNN_FFT.
    Accepts a pre-constructed fusion_module (dependency injection)
    so any fusion strategy can be benchmarked without subclassing.

    Input : (B, 1, 16, 512)  — normalised log-magnitude spectrogram
    Output: (B, N_CLASSES)   — raw logits
    """
    def __init__(self, fusion_module, quantum_type='real',
                 dropout_cnn=0.2350, dropout_skip=0.2012):
        super().__init__()
        self.quantum_type = quantum_type

        # CNN backbone (identical to HybridQNN_FFT)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(16, 32, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Flatten(),
            nn.Linear(32 * 4 * 128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_cnn),
        )

        # Quantum branch
        self.qnn_proj = nn.Sequential(
            nn.Linear(256, n_qubits),
            nn.Sigmoid(),
        )
        if quantum_type == 'complex':
            self.qnn = BatchedQuantumLayerComplex(n_layers, n_qubits, qnode_complex)
        else:
            self.qnn = BatchedQuantumLayer(n_layers, n_qubits, qnode_real)

        # Classical skip branch
        self.classical_skip = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_skip),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_skip),
        )

        self.fusion = fusion_module

    def forward(self, x):
        features  = self.cnn(x)
        qbn_input = self.qnn_proj(features) * (2.0 * math.pi)
        q_feats   = self.qnn(qbn_input)
        c_feats   = self.classical_skip(features)
        return self.fusion(c_feats, q_feats)

In [10]:
# ==========================================
# SMOKE TEST — all 8 sweep configurations
# ==========================================

# Fusion-specific dimensions
D_FUSION = 64   # shared projection dim for RFF and cross-attention
D_BP     = 32   # bilinear rank → 32*32=1024 fused features

print('Smoke testing all 8 sweep configurations...\n')
_x = X_train_t[:2].to(device)

smoke_configs = [
    ('rff',        'real',    'rbf'),
    ('rff',        'complex', 'rbf'),
    ('rff',        'real',    'laplacian'),
    ('rff',        'complex', 'laplacian'),
    ('cross_attn', 'real',    '--'),
    ('cross_attn', 'complex', '--'),
    ('bilinear',   'real',    '--'),
    ('bilinear',   'complex', '--'),
]

for fusion_type, quantum_type, kernel in smoke_configs:
    q_dim = 32 if quantum_type == 'complex' else 4
    if fusion_type == 'rff':
        fusion = RFFusion(c_dim=64, q_dim=q_dim, d_rff=D_FUSION,
                          n_classes=N_CLASSES, kernel=kernel).to(device)
    elif fusion_type == 'cross_attn':
        fusion = CrossAttnFusion(c_dim=64, q_dim=q_dim, d_model=D_FUSION,
                                 n_classes=N_CLASSES).to(device)
    else:
        fusion = BilinearFusion(c_dim=64, q_dim=q_dim, d_bp=D_BP,
                                n_classes=N_CLASSES).to(device)

    model = HybridQNN_Alt(fusion, quantum_type=quantum_type).to(device)
    with torch.no_grad():
        out = model(_x)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  {fusion_type:10s}  quantum={quantum_type:7s}  kernel={kernel:9s}  '
          f'out={list(out.shape)}  params={n_p:,}')
    del model

print('\nAll smoke tests passed.')

Smoke testing all 8 sweep configurations...

  rff         quantum=real     kernel=rbf        out=[2, 5]  params=4,247,097
  rff         quantum=complex  kernel=rbf        out=[2, 5]  params=4,248,889
  rff         quantum=real     kernel=laplacian  out=[2, 5]  params=4,247,097
  rff         quantum=complex  kernel=laplacian  out=[2, 5]  params=4,248,889
  cross_attn  quantum=real     kernel=--         out=[2, 5]  params=4,288,633
  cross_attn  quantum=complex  kernel=--         out=[2, 5]  params=4,290,425
  bilinear    quantum=real     kernel=--         out=[2, 5]  params=4,251,577
  bilinear    quantum=complex  kernel=--         out=[2, 5]  params=4,252,473

All smoke tests passed.


In [11]:
# ==========================================
# TRAINING HYPERPARAMETERS  (same as FFT notebook)
# ==========================================
lr           = 0.00079146
weight_decay = 1.11589e-06

class_counts = np.bincount(y_train_np)
cw_vals      = 1.0 / class_counts.astype(np.float32)
cw_vals      = cw_vals / cw_vals.sum() * N_CLASSES
loss_weights = torch.tensor(cw_vals, dtype=torch.float32).to(device)

MAX_EPOCHS_PER_CONFIG = 30
PATIENCE_PER_CONFIG   = 5

print(f'lr={lr}  wd={weight_decay}')
print(f'class weights: {cw_vals.tolist()}')
print(f'Per-config budget: {MAX_EPOCHS_PER_CONFIG} epochs, patience={PATIENCE_PER_CONFIG}')
print(f'D_FUSION={D_FUSION}  D_BP={D_BP}')

lr=0.00079146  wd=1.11589e-06
class weights: [1.0, 1.0, 1.0, 1.0, 1.0]
Per-config budget: 30 epochs, patience=5
D_FUSION=64  D_BP=32


In [12]:
# ==========================================
# TRAINING UTILITY
# ==========================================

def train_config_alt(fusion_type, quantum_type, kernel='rbf',
                     n_epochs=MAX_EPOCHS_PER_CONFIG, patience=PATIENCE_PER_CONFIG,
                     verbose=True):
    """Train one HybridQNN_Alt configuration and return test-set metrics."""
    set_seed(SEED)

    q_dim = 32 if quantum_type == 'complex' else 4

    if fusion_type == 'rff':
        fusion_module = RFFusion(
            c_dim=64, q_dim=q_dim, d_rff=D_FUSION,
            n_classes=N_CLASSES, kernel=kernel
        )
    elif fusion_type == 'cross_attn':
        fusion_module = CrossAttnFusion(
            c_dim=64, q_dim=q_dim, d_model=D_FUSION,
            n_heads=4, n_classes=N_CLASSES, dropout=0.1
        )
    elif fusion_type == 'bilinear':
        fusion_module = BilinearFusion(
            c_dim=64, q_dim=q_dim, d_bp=D_BP, n_classes=N_CLASSES
        )
    else:
        raise ValueError(f'Unknown fusion_type: {fusion_type}')

    model     = HybridQNN_Alt(fusion_module, quantum_type=quantum_type).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(weight=loss_weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    best_f1      = 0.0
    best_weights = None
    best_epoch   = 0
    no_impr      = 0

    for epoch in range(n_epochs):
        model.train()
        for inputs, targets in train_loader:
            inputs  = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds, all_tgts = [], []
        with torch.no_grad():
            for inputs, targets in test_loader:
                out = model(inputs.to(device, non_blocking=True))
                all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                all_tgts.extend(targets.numpy())

        f1 = f1_score(all_tgts, all_preds, average='macro', zero_division=0)
        scheduler.step(f1)

        if f1 > best_f1:
            best_f1      = f1
            best_weights = copy.deepcopy(model.state_dict())
            best_epoch   = epoch + 1
            no_impr      = 0
        else:
            no_impr += 1

        if verbose:
            mark = ' \u2190' if no_impr == 0 else ''
            print(f'  ep {epoch+1:02d}  f1={f1:.4f}{mark}')

        if no_impr >= patience:
            break

    model.load_state_dict(best_weights)
    model.eval()
    all_preds, all_tgts = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            out = model(inputs.to(device))
            all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            all_tgts.extend(targets.numpy())

    return {
        'val_acc':      accuracy_score(all_tgts, all_preds),
        'macro_f1':     f1_score(all_tgts, all_preds, average='macro',    zero_division=0),
        'weighted_f1':  f1_score(all_tgts, all_preds, average='weighted', zero_division=0),
        'best_epoch':   best_epoch,
        'total_epochs': epoch + 1,
    }

In [13]:
# ==========================================
# 8-CONFIG SWEEP
# ==========================================
SWEEP = [
    # RFF — two kernels x two quantum types = 4 configs
    {'fusion_type': 'rff',        'quantum_type': 'real',    'kernel': 'rbf'},
    {'fusion_type': 'rff',        'quantum_type': 'complex', 'kernel': 'rbf'},
    {'fusion_type': 'rff',        'quantum_type': 'real',    'kernel': 'laplacian'},
    {'fusion_type': 'rff',        'quantum_type': 'complex', 'kernel': 'laplacian'},
    # Cross-attention — two quantum types = 2 configs
    {'fusion_type': 'cross_attn', 'quantum_type': 'real',    'kernel': '--'},
    {'fusion_type': 'cross_attn', 'quantum_type': 'complex', 'kernel': '--'},
    # Bilinear — two quantum types = 2 configs
    {'fusion_type': 'bilinear',   'quantum_type': 'real',    'kernel': '--'},
    {'fusion_type': 'bilinear',   'quantum_type': 'complex', 'kernel': '--'},
]

print(f'Running {len(SWEEP)} configurations...\n')
for cfg in SWEEP:
    print(f"  fusion={cfg['fusion_type']:10s}  quantum={cfg['quantum_type']:7s}  kernel={cfg['kernel']}")
print(f'\nEpoch budget : {MAX_EPOCHS_PER_CONFIG} / patience {PATIENCE_PER_CONFIG}')
print(f'FFT best (ref): complex + product + gelu  -> 0.9717 macro F1')
print(f'FiLM baseline : 0.9566 macro F1\n')

alt_results = []
t_total     = time.time()

for i, cfg in enumerate(SWEEP, 1):
    ft = cfg['fusion_type']
    qt = cfg['quantum_type']
    k  = cfg['kernel']

    label = f'[{i:02d}/{len(SWEEP)}]  fusion={ft:10s}  quantum={qt:7s}  kernel={k}'
    print('=' * 65)
    print(label)
    print('=' * 65)
    t0 = time.time()

    metrics = train_config_alt(
        fusion_type  = ft,
        quantum_type = qt,
        kernel       = k if k != '--' else 'rbf',
        verbose      = True,
    )

    elapsed = time.time() - t0
    print(f'  -> val_acc={metrics["val_acc"]:.4f}  '
          f'macro_f1={metrics["macro_f1"]:.4f}  '
          f'best_epoch={metrics["best_epoch"]}  '
          f'({elapsed:.0f}s)\n')

    alt_results.append({
        'fusion_type':  ft,
        'quantum_type': qt,
        'kernel':       k,
        **metrics,
    })

print(f'Sweep complete in {time.time() - t_total:.0f}s')

Running 8 configurations...

  fusion=rff         quantum=real     kernel=rbf
  fusion=rff         quantum=complex  kernel=rbf
  fusion=rff         quantum=real     kernel=laplacian
  fusion=rff         quantum=complex  kernel=laplacian
  fusion=cross_attn  quantum=real     kernel=--
  fusion=cross_attn  quantum=complex  kernel=--
  fusion=bilinear    quantum=real     kernel=--
  fusion=bilinear    quantum=complex  kernel=--

Epoch budget : 30 / patience 5
FFT best (ref): complex + product + gelu  -> 0.9717 macro F1
FiLM baseline : 0.9566 macro F1

[01/8]  fusion=rff         quantum=real     kernel=rbf
  ep 01  f1=0.5866 ←
  ep 02  f1=0.7733 ←
  ep 03  f1=0.9167 ←
  ep 04  f1=0.9141
  ep 05  f1=0.9433 ←
  ep 06  f1=0.9516 ←
  ep 07  f1=0.9467
  ep 08  f1=0.9367
  ep 09  f1=0.9500
  ep 10  f1=0.9365
  ep 11  f1=0.9549 ←
  ep 12  f1=0.9567 ←
  ep 13  f1=0.9616 ←
  ep 14  f1=0.9567
  ep 15  f1=0.9533
  ep 16  f1=0.9633 ←
  ep 17  f1=0.9483
  ep 18  f1=0.9517
  ep 19  f1=0.9517
  ep 20  f1

In [14]:
# ==========================================
# RESULTS & COMPARISON TABLE
# ==========================================
df_alt = pd.DataFrame(alt_results)
df_alt = df_alt.sort_values('macro_f1', ascending=False).reset_index(drop=True)
df_alt.index = df_alt.index + 1

fmt = {'val_acc': '{:.4f}', 'macro_f1': '{:.4f}', 'weighted_f1': '{:.4f}'}
df_disp = df_alt.copy()
for col, f in fmt.items():
    df_disp[col] = df_alt[col].apply(lambda v: f.format(v))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 130)

print('\n' + '=' * 90)
print('ALTERNATIVE FUSION METHODS — RESULTS  (sorted by Macro F1, best -> worst)')
print('=' * 90)
print(df_disp[['fusion_type', 'quantum_type', 'kernel',
               'val_acc', 'macro_f1', 'weighted_f1',
               'best_epoch', 'total_epochs']].to_string())

print('\n' + '-' * 90)
print('REFERENCE ROWS (from other notebooks):')
print(f'  FFT best  : fusion=fft        quantum=complex  kernel=product  macro_f1=0.9717')
print(f'  FiLM base : fusion=film       quantum=real     kernel=--       macro_f1=0.9566')

# Grouped summary by fusion method
print('\n' + '=' * 90)
print('MEAN MACRO F1 BY FUSION METHOD')
print('=' * 90)
grp = df_alt.groupby('fusion_type')['macro_f1'].agg(['mean', 'max', 'min']).round(4)
print(grp.to_string())

print('\n--- All results vs FFT best (0.9717) and FiLM baseline (0.9566) ---')
for _, row in df_alt.iterrows():
    vs_fft  = row['macro_f1'] - 0.9717
    vs_film = row['macro_f1'] - 0.9566
    print(f"  {row['fusion_type']:10s}  {row['quantum_type']:7s}  {row['kernel']:9s}  "
          f"macro_f1={row['macro_f1']:.4f}  "
          f"vs_fft={vs_fft:+.4f}  vs_film={vs_film:+.4f}")

# Save
df_alt.to_csv(OUT_DIR / 'fusion_alternatives_results.csv', index_label='rank')
print(f'\nSaved: {OUT_DIR}/fusion_alternatives_results.csv')


ALTERNATIVE FUSION METHODS — RESULTS  (sorted by Macro F1, best -> worst)
  fusion_type quantum_type     kernel val_acc macro_f1 weighted_f1  best_epoch  total_epochs
1  cross_attn         real         --  0.9700   0.9699      0.9699           8            13
2    bilinear         real         --  0.9667   0.9667      0.9667          16            21
3         rff         real        rbf  0.9633   0.9633      0.9633          16            21
4  cross_attn      complex         --  0.9550   0.9550      0.9550           6            11
5    bilinear      complex         --  0.9500   0.9500      0.9500          13            18
6         rff      complex        rbf  0.9483   0.9483      0.9483           4             9
7         rff         real  laplacian  0.2417   0.2375      0.2375           5            10
8         rff      complex  laplacian  0.2300   0.2303      0.2303           7            12

---------------------------------------------------------------------------------------